# 5. Predicting leadership-annual-giving upgrades

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilanthroPy-Project/PhilanthroPy/blob/main/examples/notebooks/05_leadership_upgrade.ipynb)

A mid-level annual donor giving $100-$999 a year is a different prospect than
someone giving $50: they are already engaged, and some of them are one good ask
away from a leadership-level gift. This notebook builds a model that ranks
mid-level donors by how likely they are to cross that leadership threshold
next fiscal year, using `build_upgrade_snapshots` to assemble the training
table and `FiscalYearGroupedSplitter` to validate it without letting a future
fiscal year leak into training.

**This is a smoke test on synthetic data, not a claim that the model beats
any baseline on a real fundraising file.** The donor panel and the activity
log below are both generated, with a numeric signal built in on purpose so
the notebook has something to show; treat every number here as a
demonstration of the mechanism, not a result to quote for your own program.


In [ ]:
# Colab and other fresh environments only; a local checkout already has it.
try:
    import philanthropy
except ImportError:
    !pip install -q philanthropy
    import philanthropy

print("philanthropy", philanthropy.__version__)


## Build the training panel

`make_donor_panel` gives us a gift log and a donor table, but no activity log:
real shops track event attendance and volunteer shifts separately, so we build
our own small synthetic one here. Engagement (event and volunteer counts) is
drawn with a rate tied to each donor's lifetime giving, so it carries some of
the same signal as the gift history, just measured a different way and with
independent noise. That is the point: a single fiscal year's total is a noisy
read on a donor's true giving capacity, and a second, independently noisy
signal lets the model average out some of that noise instead of leaning on
one number the way the $500-last-FY rule does.


In [ ]:
import numpy as np
import pandas as pd

from philanthropy.datasets import make_donor_panel

N_YEARS = 7
START_FY = 2018

panel = make_donor_panel(
    n_donors=3000, n_years=N_YEARS, start_fiscal_year=START_FY, random_state=0
)
gifts, donors = panel["gifts"], panel["donors"]

rng = np.random.default_rng(0)

# A rough, generation-time-only proxy for each donor's underlying generosity:
# lifetime giving across the whole panel. Real activity data would obviously
# correlate with something like this; the model itself never sees it, only
# the per-fiscal-year features build_upgrade_snapshots computes from data
# through the end of each snapshot year.
lifetime_total = (
    gifts.groupby("donor_id")["gift_amount"].sum()
    .reindex(donors["donor_id"], fill_value=0.0)
)
engagement_percentile = lifetime_total.rank(pct=True).to_numpy()

event_lambda = (0.3 + 2.0 * engagement_percentile) * N_YEARS
volunteer_lambda = (0.1 + 1.0 * engagement_percentile) * N_YEARS
n_events = rng.poisson(event_lambda)
n_volunteer = rng.poisson(volunteer_lambda)

window_start = pd.Timestamp(f"{START_FY - 1}-07-01")
window_days = 365 * N_YEARS


def activity_rows(donor_ids, counts, activity_type, with_hours=False):
    donor_id_rep = np.repeat(donor_ids, counts)
    dates = window_start + pd.to_timedelta(
        rng.integers(0, window_days, size=len(donor_id_rep)), unit="D"
    )
    rows = {
        "contact_id": donor_id_rep,
        "activity_date": dates,
        "activity_type": activity_type,
    }
    if with_hours:
        rows["hours"] = rng.integers(1, 5, size=len(donor_id_rep))
    return pd.DataFrame(rows)


donor_ids = donors["donor_id"].to_numpy()
activities = pd.concat(
    [
        activity_rows(donor_ids, n_events, "event"),
        activity_rows(donor_ids, n_volunteer, "volunteer", with_hours=True),
    ],
    ignore_index=True,
)
activities.head()


## Build upgrade-candidate snapshots

`build_upgrade_snapshots` turns the gift log into one row per (donor, fiscal
year T) for every donor whose FY T giving falls in the $100-$999 band: the
mid-level donors a leadership-gift officer would actually work. `target` is 1
if that donor's FY T+1 total reaches $1,000; everything else is computed
from data through the end of FY T, so nothing about the outcome leaks into the
features. Passing `activities` and `donors` joins in the engagement features
and the (partly missing) wealth estimate alongside the gift-derived ones.


In [ ]:
from philanthropy.ingest import build_upgrade_snapshots

snapshots = build_upgrade_snapshots(
    gifts,
    fiscal_years=range(START_FY + 1, START_FY + N_YEARS - 1),
    threshold=1000,
    band=(100, 999),
    activities=activities,
    donors=donors.set_index("donor_id"),
)

print(snapshots.shape, "rows")
print("upgrade rate:", round(snapshots["target"].mean(), 3))
snapshots.head()


## Fit with a fiscal-year walk-forward split

"Did this donor upgrade in FY T+1?" is a time-varying label, not a static
per-donor one: a donor who shows up as a candidate in several fiscal years can
legitimately sit in both an earlier training fold and a later test fold,
because each row's target is read from a different, later year.
`FiscalYearGroupedSplitter` with `drop_repeat_donors=False` is the walk-forward
mode built for exactly that case, training on every fiscal year strictly
before the test year and never the reverse.


In [ ]:
from philanthropy.model_selection import FiscalYearGroupedSplitter
from philanthropy.models import MajorGiftClassifier

feature_cols = [
    c
    for c in snapshots.select_dtypes(include="number").columns
    if c not in ("target", "fiscal_year")
]
X = snapshots[feature_cols]
y = snapshots["target"].to_numpy()
fiscal_year = snapshots["fiscal_year"].to_numpy()

splitter = FiscalYearGroupedSplitter(n_splits=3, drop_repeat_donors=False)

oof_scores, oof_targets, oof_fy_total = [], [], []
for train_idx, test_idx in splitter.split(X, groups=fiscal_year):
    model = MajorGiftClassifier(random_state=0).fit(X.iloc[train_idx], y[train_idx])
    oof_scores.append(model.predict_affinity_score(X.iloc[test_idx]))
    oof_targets.append(y[test_idx])
    oof_fy_total.append(X.iloc[test_idx]["fy_total"].to_numpy())

scores = np.concatenate(oof_scores)
targets = np.concatenate(oof_targets)
fy_total = np.concatenate(oof_fy_total)
print(f"{len(targets)} out-of-fold predictions across {splitter.get_n_splits(X, groups=fiscal_year)} walk-forward folds")


## Model vs. the $500-last-FY rule

The naive rule a gift officer might use without a model: flag anyone who
already gave $500 or more this fiscal year (`fy_total >= 500`) as an upgrade
prospect. We compare that against the model's own top 20% by affinity score,
on the same out-of-fold predictions, so neither side is scored on data it was
trained on.


In [ ]:
overall_rate = targets.mean()

top_n = max(1, int(round(0.20 * len(targets))))
model_top_idx = np.argsort(-scores)[:top_n]
model_rate = targets[model_top_idx].mean()

rule_flag = fy_total >= 500
rule_rate = targets[rule_flag].mean()

rule_top_idx = np.argsort(-fy_total)[:top_n]
rule_top_rate = targets[rule_top_idx].mean()

pd.DataFrame(
    {
        "group": [
            "overall (all candidates)",
            f"model, top {top_n} ({top_n / len(targets):.0%}) by affinity score",
            f"rule, everyone flagged (fy_total >= $500, n={int(rule_flag.sum())})",
            f"rule, its own top {top_n} by fy_total",
        ],
        "upgrade rate": [overall_rate, model_rate, rule_rate, rule_top_rate],
    }
).round(3)


## Smoke test, not a benchmark

The model's top 20% upgrades at a noticeably higher rate than either version
of the $500 rule, and both beat the overall base rate, because the generator
above was built to give the model more than one noisy signal to combine.
These are synthetic numbers on a generator whose engagement-giving correlation
was chosen; they say nothing about how this model would do on a real donor
file, and should not be quoted as evidence it beats any particular baseline
in production. Re-validate on your own fiscal-year history before trusting a
ranking like this one for real leadership-gift outreach.
